In [1]:
from tqdm import tqdm
from dataset import MyriadLamaDataset

model_name = "qwen3_8b"
dataset = MyriadLamaDataset(model_name=model_name, debug=True)

Debug mode: using a smaller subset of the dataset.
Dataset already exists at /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen3_8b/paraphrases_dataset. Loading from disk.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from constants import MODEL_PATHs

model_name = MODEL_PATHs[model_name]
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else None,
    device_map="cuda:1",
        
).eval()


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
import json
import re
from typing import Dict, Tuple, Optional

def extract_json(text: str) -> Optional[Dict]:
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def validate_paraphrase(
    src: str,
    obj: Dict,
) -> Tuple[bool, str]:
    if not isinstance(obj, dict):
        return False, "Output is not a JSON object."

    if "paraphrase" not in obj or not isinstance(obj["paraphrase"], str):
        return False, "Missing string field: paraphrase."

    para = obj["paraphrase"].strip()

    if len(para) < 3:
        return False, "Paraphrase is too short."

    # Simple constraints you can hard-enforce:
    # 1) must not be identical
    if para == src.strip():
        return False, "Paraphrase is identical to source."

    # 2) preserve numbers (hard rule)
    src_nums = re.findall(r"\d+(?:\.\d+)?", src)
    para_nums = re.findall(r"\d+(?:\.\d+)?", para)
    if src_nums != para_nums:
        return False, f"Numbers not preserved. src={src_nums}, para={para_nums}"

    # # 3) one sentence (optional, naive)
    # if para.count(".") + para.count("!") + para.count("?") > 1:
    #     return False, "Paraphrase must be one sentence."

    # 4) Assure [MASK] in the generation
    if "[MASK]" in src and "[MASK]" not in para:
        return False, "[MASK] token missing in paraphrase."
    
    return True, "OK"


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class ParaphraseExtract(BaseModel):
    paraphrase: str = Field(
        description="A Boolean value indicating whether the sentence contains generalizable triple-like factual knowledge."
    )

parser = PydanticOutputParser(pydantic_object=ParaphraseExtract, include_raw=True, strict=False)

def extract_first_json_block(text: str) -> str:
    """
    Extract the first valid JSON object from a string with nested braces.
    """
    start = text.find("{")
    if start == -1:
        raise ValueError("No opening brace found in text")

    brace_count = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            brace_count += 1
        elif text[i] == "}":
            brace_count -= 1
            if brace_count == 0:
                return text[start:i+1]
    raise ValueError("Braces do not match, incomplete JSON block")

def robust_parse(output_str):
    """
    Preprocess LLM output string and try to parse it robustly using a LangChain parser.
    """
    cleaned = re.sub(r"^```(?:json)?|```$", "", output_str.strip(), flags=re.MULTILINE).strip()

    # Normalize casing for JSON literals
    cleaned = re.sub(r'\bNULL\b', 'null', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\bTRUE\b', 'true', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\bFALSE\b', 'false', cleaned, flags=re.IGNORECASE)

    try:
        json_block = extract_first_json_block(cleaned)
    except ValueError:
        return None
    
    try:
        return parser.parse(json_block).model_dump()
    except Exception:
        return None
    

In [14]:
from collections import Counter
import torch

def generate_text_batch(model, tokenizer, prompts: list[str], top_p, max_new_tokens, temperature) -> list[str]:
    """
    Generates text for a batch of prompts efficiently.
    """
    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    input_len = inputs.input_ids.shape[1]
    generated_tokens = out[:, input_len:]
    decoded_texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return [text.strip() for text in decoded_texts]

def enforced_paraphrase(
        prompt, 
        batch_size, 
        temperature, top_p,
        num_paraphrase, max_rounds, 
        verbose=False):
    system_rules = (
        "You are a paraphrasing engine.\n"
        "You MUST follow the output contract exactly.\n"
        "Return ONLY valid JSON. No extra words.\n"
        "Strategies to apply:\n"
        "- Lexicon Replacement: Swap words for synonyms while maintaining nuance.\n"
        "- Syntax Reordering: Change grammatical structure (e.g., active/passive, clause order).\n"
        "- Semantic Restructuring: Express the same truth conditions using different wording logic (logical entailment).\n"
        "Constraints:\n"
        "- Maintain Strict Entailment: The paraphrase must be true if and only if the source is true.\n"
        "- Preserve ALL numbers/named entities exactly.\n"
        "- One sentence.\n"
        "- Do not add new facts.\n"
        "\nBelow are some examples: \n"
        "For the given prompts: Which continent is Queen Maud Land situated on? [MASK]\n"
        "Lexicon Replacement paraphrase: In which continent can Queen Maud Land be found? [MASK]\n"
        "Syntax Reordering paraphrase: Queen Maud Land is a region located on [MASK] continent.\n"
        "Semantic Restructuring paraphrase: Identify the continental landmass that contains Queen Maud Land: [MASK]"
    )

    contract = (
        'Output JSON schema:\n'
        '{\n'
        '  "paraphrase": string,\n'
        '}\n'
    )

    def build_prompt(user_msg: str) -> str:
        if hasattr(tokenizer, "apply_chat_template"):
            messages = [
                {"role": "system", "content": system_rules + "\n" + contract},
                {"role": "user", "content": user_msg},
            ]
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        return system_rules + "\n" + contract + "\nUSER:\n" + user_msg + "\nASSISTANT:\n"

    collected_paraphrases = []
    seen_texts = set()
    
    user_msg_base = f"Source: {prompt}\nTask: paraphrase the Source under the Constraints."
    query_text = build_prompt(user_msg_base)
    
    # pbar = tqdm(total=num_paraphrase, desc="Collecting paraphrases")
    # pbar = tqdm(total=num_paraphrase)
    parse_error, valid_error, duplicated_error = 0, 0, 0
    valid_error_details = Counter()
    for round_idx in range(max_rounds):
        if len(collected_paraphrases) >= num_paraphrase:
            break
        
        current_batch_inputs = [query_text] * batch_size
        raw_outputs = generate_text_batch(
            model, 
            tokenizer, 
            current_batch_inputs, 
            top_p=top_p,
            max_new_tokens=256, 
            temperature=temperature
        )

        for raw in raw_outputs:
            if len(collected_paraphrases) >= num_paraphrase:
                break

            obj = robust_parse(raw)
            if not obj:
                parse_error += 1
                continue

            ok, reason = validate_paraphrase(prompt, obj)
            valid_error_details[reason] += 1
            if not ok:
                valid_error += 1
                continue
            candidate = obj["paraphrase"].strip()
            if candidate != prompt and candidate not in seen_texts:
                seen_texts.add(candidate)
                collected_paraphrases.append(candidate)
            else:
                duplicated_error += 1

    if verbose:
        print(f"Parsing errors: {parse_error}, Validation errors: {valid_error}, Duplicated paraphrases: {duplicated_error}")
        print("Validation error details:", dict(valid_error_details))
    return raw_outputs, collected_paraphrases[:num_paraphrase]

In [ ]:
import os

from utils import dump_jsonl, load_jsonl

verbose = False
num_paraphrase = 2
max_rounds = 2
temperature = 1.5
topp = 0.95
dataloader = dataset.get_dataloader(batch_size=1, shuffle=False)
    
paraphrase_path = os.path.join(dataset.dataset_path, f"paraphrases.num_para{num_paraphrase}.max_rounds{max_rounds}.temp{temperature}.topp{topp}.jsonl")

if os.path.exists(paraphrase_path):
    paraphrases = {item["uuid"]: item for item in load_jsonl(paraphrase_path, verbose=True)}
else:
    paraphrases = {}

for idx, (uuids, answers, all_paraphrases, _) in tqdm(enumerate(dataloader)):
    uuid = uuids[0]
    if uuid in paraphrases:
        if verbose:
            print(f"Skipping existing UUID: {uuid}")
        continue

    answer = answers[0]
    seed_prompt = all_paraphrases[0][0]
    raw_outputs, _paraphrases = enforced_paraphrase(
        seed_prompt, 
        batch_size=8, 
        temperature=temperature, top_p=topp,
        num_paraphrase=num_paraphrase, max_rounds=max_rounds, verbose=False)
    
    if verbose:
        print(f"-------- UUID: {uuid} --------")
        print("Seed prompt:", seed_prompt)
        print("Number of paraphrases:", len(_paraphrases))
        print("Paraphrases:", _paraphrases)
    paraphrases[uuid] = {
        "uuid": uuid,
        "answers": answer,
        "seed_prompt": seed_prompt,
        "auto_paraphrases": _paraphrases,
    }
    
dump_jsonl(paraphrases.values(), paraphrase_path)

0it [00:01, ?it/s]


In [ ]:
dataset.dataset_root

'/home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen3_8b'